Notebook 02 — Détection des biais de représentativité=======================================================Projet : Détection de biais dans les données de santé au travailAuteure : Laure Bonnefond — IPRP / IST — Master IA Data IPSSI BordeauxCe notebook analyse la représentativité démographique (sexe, âge) desdonnées AT/MP par rapport à la population active réelle (INSEE).Méthodes :  - Ratio de représentation  - Test du Chi² d'ajustement  - Indice de diversité de Shannon  - Matrice de biais (heatmap)Objectif IA responsable :  Identifier si un modèle entraîné sur ces données reproduirait des  inégalités de prédiction selon le sexe ou l'âge des salariés.

# 1. Configuration


In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.utils import (
    ratio_representation,
    test_chi2_representation,
    indice_shannon,
    indice_equitabilite,
    PALETTE,
)

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

print("Environnement prêt — Notebook 02 : Détection des biais")


# 2. Chargement des données


In [ ]:
df_atmp = pd.read_csv('../data/raw/atmp_par_secteur.csv')
df_insee = pd.read_csv('../data/raw/insee_population_active_secteur.csv')

# Fusion sur la clé CTN
df = df_atmp.merge(df_insee, on=['ctn', 'secteur'], suffixes=('', '_insee'))
print(f"Données fusionnées : {df.shape[0]} secteurs × {df.shape[1]} colonnes")
df[['ctn', 'secteur', 'pct_hommes', 'pct_hommes_insee',
    'pct_femmes', 'pct_femmes_insee']].head()


# 3. Analyse du biais de genre (sexe)

**Question** : Les femmes sont-elles correctement représentées dans
les données AT/MP par rapport à leur présence réelle dans chaque secteur ?


## 3.1 Calcul des ratios de représentation


In [ ]:
df['ratio_hommes'] = df['pct_hommes'] / df['pct_hommes_insee']
df['ratio_femmes'] = df['pct_femmes'] / df['pct_femmes_insee']

print("RATIOS DE REPRÉSENTATION PAR SEXE")
print("=" * 60)
print(f"{'Secteur':<35} {'Ratio H':>10} {'Ratio F':>10}")
print("-" * 60)
for _, row in df.iterrows():
    flag_h = "⚠️" if abs(row['ratio_hommes'] - 1) > 0.1 else "✅"
    flag_f = "⚠️" if abs(row['ratio_femmes'] - 1) > 0.1 else "✅"
    print(f"{row['secteur'][:34]:<35} "
          f"{row['ratio_hommes']:>7.2f} {flag_h}  "
          f"{row['ratio_femmes']:>7.2f} {flag_f}")

print("\nLecture :")
print("  Ratio = 1.0  → représentation parfaite")
print("  Ratio < 1.0  → sous-représentation")
print("  Ratio > 1.0  → surreprésentation")


## 3.2 Visualisation du biais de genre


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Graphique 1 : Comparaison directe
x = np.arange(len(df))
width = 0.35
axes[0].barh(x - width/2, df['pct_hommes'], width,
             label='AT/MP', color='#185FA5')
axes[0].barh(x + width/2, df['pct_hommes_insee'], width,
             label='Population active', color='#85B7EB')
axes[0].set_yticks(x)
axes[0].set_yticklabels(df['secteur'], fontsize=9)
axes[0].set_xlabel('% Hommes')
axes[0].set_title('% Hommes : AT/MP vs Population active')
axes[0].legend()
axes[0].set_xlim(0, 100)

# Graphique 2 : Ratios
colors_h = ['#E24B4A' if r < 0.9 else '#BA7517' if r < 0.95
            else '#0F6E56' if r <= 1.05 else '#185FA5'
            for r in df['ratio_femmes']]
axes[1].barh(df['secteur'], df['ratio_femmes'], color=colors_h)
axes[1].axvline(x=1, color='black', linestyle='--', linewidth=1,
                label='Représentation parfaite')
axes[1].axvline(x=0.9, color='red', linestyle=':', linewidth=1, alpha=0.6,
                label='Seuil sous-représentation (-10%)')
axes[1].axvline(x=1.1, color='red', linestyle=':', linewidth=1, alpha=0.6)
axes[1].set_xlabel('Ratio de représentation des femmes')
axes[1].set_title("Biais de représentation des femmes dans les AT/MP")
axes[1].legend(loc='lower right', fontsize=9)
for i, val in enumerate(df['ratio_femmes']):
    axes[1].text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/02_biais_genre.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/02_biais_genre.png")


## 3.3 Test statistique du Chi² par secteur

Pour chaque secteur, on teste si la distribution H/F observée dans les
AT/MP diffère significativement de la distribution attendue (INSEE).


In [ ]:
print("TEST DU CHI² — DISTRIBUTION H/F PAR SECTEUR")
print("=" * 70)
print(f"{'Secteur':<35} {'Chi²':>8} {'p-value':>10} {'Biais':>10}")
print("-" * 70)

resultats_chi2 = []
for _, row in df.iterrows():
    nb_at = row['nb_at']
    # Effectifs observés dans les AT/MP
    obs = [
        nb_at * row['pct_hommes'] / 100,
        nb_at * row['pct_femmes'] / 100,
    ]
    # Effectifs attendus selon INSEE
    exp = [
        nb_at * row['pct_hommes_insee'] / 100,
        nb_at * row['pct_femmes_insee'] / 100,
    ]
    chi2, p_value = stats.chisquare(f_obs=obs, f_exp=exp)
    significatif = p_value < 0.05
    flag = "⚠️ Oui" if significatif else "✅ Non"
    print(f"{row['secteur'][:34]:<35} {chi2:>8.1f} {p_value:>10.4f}   {flag}")
    resultats_chi2.append({
        'secteur': row['secteur'],
        'chi2': chi2,
        'p_value': p_value,
        'biais_significatif': significatif,
    })

df_chi2 = pd.DataFrame(resultats_chi2)
nb_biais = df_chi2['biais_significatif'].sum()
print("-" * 70)
print(f"Total : {nb_biais}/{len(df_chi2)} secteurs présentent un biais significatif (p < 0.05)")


# 4. Analyse du biais d'âge


## 4.1 Construction de la matrice des ratios par âge


In [ ]:
tranches = ['18_29', '30_44', '45_59', '60_plus']
labels_tranches = ['18-29 ans', '30-44 ans', '45-59 ans', '60+ ans']

matrice_ratios = pd.DataFrame(index=df['secteur'])
for tr, label in zip(tranches, labels_tranches):
    matrice_ratios[label] = (df[f'pct_{tr}'].values /
                              df[f'pct_{tr}_insee'].values).round(2)

print("MATRICE DES RATIOS DE REPRÉSENTATION PAR TRANCHE D'ÂGE")
print("=" * 70)
print(matrice_ratios.to_string())
print("\nLecture : 1.00 = parfait, <1 = sous-représenté, >1 = surreprésenté")


## 4.2 Heatmap des biais d'âge


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))

# Centrer la heatmap sur 1.0 (représentation parfaite)
sns.heatmap(matrice_ratios, annot=True, fmt='.2f',
            cmap='RdYlGn', center=1.0, vmin=0.5, vmax=1.5,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Ratio de représentation (1.0 = parfait)'})
ax.set_title("Biais de représentation par âge dans les AT/MP\n"
             "(données AT/MP vs population active INSEE)")
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../assets/02_heatmap_biais_age.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/02_heatmap_biais_age.png")


## 4.3 Identification des biais critiques


In [ ]:
print("BIAIS CRITIQUES PAR ÂGE (écart > 10%)")
print("=" * 60)

biais_critiques = []
for secteur in matrice_ratios.index:
    for tranche in matrice_ratios.columns:
        ratio = matrice_ratios.loc[secteur, tranche]
        if abs(ratio - 1) > 0.10:
            type_biais = "Surreprésentation" if ratio > 1 else "Sous-représentation"
            ecart_pct = (ratio - 1) * 100
            biais_critiques.append({
                'secteur': secteur,
                'tranche_age': tranche,
                'ratio': ratio,
                'ecart_pct': ecart_pct,
                'type': type_biais,
            })

df_biais = pd.DataFrame(biais_critiques).sort_values('ecart_pct', key=abs, ascending=False)
print(f"\n{len(df_biais)} biais détectés (sur {9*4} combinaisons possibles)\n")
print(df_biais.to_string(index=False))


# 5. Indices de diversité (Shannon)

L'indice de Shannon mesure la diversité d'une distribution.
Plus il est élevé, plus la distribution est équilibrée.


In [ ]:
print("INDICES DE DIVERSITÉ DE SHANNON")
print("=" * 70)
print(f"{'Secteur':<35} {'Shannon AT/MP':>15} {'Shannon INSEE':>15}")
print("-" * 70)

resultats_shannon = []
for _, row in df.iterrows():
    # Distribution âge AT/MP
    props_atmp = [row[f'pct_{t}'] / 100 for t in tranches]
    # Distribution âge INSEE
    props_insee = [row[f'pct_{t}_insee'] / 100 for t in tranches]
    
    h_atmp = indice_shannon(props_atmp)
    h_insee = indice_shannon(props_insee)
    
    print(f"{row['secteur'][:34]:<35} {h_atmp:>15.3f} {h_insee:>15.3f}")
    resultats_shannon.append({
        'secteur': row['secteur'],
        'shannon_atmp': h_atmp,
        'shannon_insee': h_insee,
        'perte_diversite': h_insee - h_atmp,
    })

df_shannon = pd.DataFrame(resultats_shannon)
print("-" * 70)
print(f"Moyenne perte de diversité : {df_shannon['perte_diversite'].mean():+.3f}")
print("(positif = AT/MP moins diversifié que la population réelle)")


# 6. Synthèse globale du biais


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Score de biais global par secteur
df['score_biais_age'] = matrice_ratios.apply(
    lambda row: np.mean([abs(v - 1) for v in row]), axis=1).values
df['score_biais_genre'] = (df['ratio_femmes'] - 1).abs()
df['score_biais_total'] = (df['score_biais_age'] + df['score_biais_genre']) / 2

df_sorted = df.sort_values('score_biais_total', ascending=True)
colors = ['#0F6E56' if s < 0.05 else '#BA7517' if s < 0.10 else '#E24B4A'
          for s in df_sorted['score_biais_total']]
axes[0, 0].barh(df_sorted['secteur'], df_sorted['score_biais_total'] * 100,
                color=colors)
axes[0, 0].set_xlabel('Score de biais global (%)')
axes[0, 0].set_title('Score de biais global par secteur\n(moyenne des écarts âge + genre)')
for i, val in enumerate(df_sorted['score_biais_total']):
    axes[0, 0].text(val * 100 + 0.3, i, f'{val*100:.1f}%',
                    va='center', fontsize=9)

# 2. Comparaison Shannon
x_pos = np.arange(len(df_shannon))
width = 0.35
axes[0, 1].bar(x_pos - width/2, df_shannon['shannon_atmp'], width,
               label='AT/MP', color='#D85A30')
axes[0, 1].bar(x_pos + width/2, df_shannon['shannon_insee'], width,
               label='INSEE', color='#0F6E56')
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels([f"CTN {c}" for c in df_shannon.index.map(
    lambda i: df.iloc[i]['ctn'])], rotation=0)
axes[0, 1].set_ylabel('Indice de Shannon (diversité d\'âge)')
axes[0, 1].set_title('Diversité d\'âge : AT/MP vs Population active')
axes[0, 1].legend()

# 3. Top 5 biais critiques
df_top = df_biais.head(8).copy()
colors_b = ['#E24B4A' if t == 'Sous-représentation' else '#185FA5'
            for t in df_top['type']]
axes[1, 0].barh(
    [f"{s[:20]}\n{t}" for s, t in zip(df_top['secteur'], df_top['tranche_age'])],
    df_top['ecart_pct'], color=colors_b)
axes[1, 0].axvline(x=0, color='black', linewidth=0.8)
axes[1, 0].set_xlabel('Écart de représentation (%)')
axes[1, 0].set_title('Top 8 biais démographiques détectés')
axes[1, 0].set_yticklabels(axes[1, 0].get_yticklabels(), fontsize=8)

# 4. Tests Chi²
colors_chi2 = ['#E24B4A' if b else '#0F6E56'
               for b in df_chi2['biais_significatif']]
axes[1, 1].barh(df_chi2['secteur'], -np.log10(df_chi2['p_value'] + 1e-10),
                color=colors_chi2)
axes[1, 1].axvline(x=-np.log10(0.05), color='black', linestyle='--',
                   linewidth=1, label='Seuil p=0.05')
axes[1, 1].set_xlabel('-log₁₀(p-value)')
axes[1, 1].set_title('Significativité statistique du biais de genre')
axes[1, 1].legend()

plt.suptitle('Synthèse globale des biais détectés dans les données AT/MP',
             fontsize=16, y=1.00)
plt.tight_layout()
plt.savefig('../assets/02_synthese_biais.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée : assets/02_synthese_biais.png")


# 7. Interprétation métier (IPRP)

**Pourquoi ces biais existent-ils ?** L'analyse statistique seule ne suffit
pas — il faut une lecture métier pour comprendre les mécanismes en jeu :

### Biais structurels du système déclaratif

- **Sous-déclaration des femmes dans le BTP/Métallurgie** : les femmes
  occupent souvent des postes administratifs moins exposés, donc le taux
  d'AT déclaré est mécaniquement plus faible — mais cela peut aussi
  masquer des AT non déclarés pour ces postes "périphériques".

- **Surreprésentation des seniors** dans certains secteurs : les TMS et MP
  apparaissent en fin de carrière (latence longue), donc les 45-59 ans
  sont surreprésentés dans les MP.

- **Effet "travailleur sain"** : les salariés les plus fragiles quittent
  les secteurs à risque (sortie précoce), ce qui sous-estime les risques
  réels pour les seniors.

### Implications pour un modèle prédictif

Si on entraîne un modèle ML sur ces données sans correction, il risque de :

1. **Sous-estimer le risque pour les femmes** dans les secteurs industriels
2. **Mal généraliser aux jeunes** sous-représentés dans les MP
3. **Reproduire les biais de déclaration** comme s'ils étaient des faits

C'est exactement le problème pointé par les chercheurs en IA responsable :
*"Un modèle entraîné sur des données biaisées peut reproduire des inégalités
sans que personne ne s'en rende compte."*

### Pistes de mitigation (Notebook 04)

- Repondération des observations (inverse propensity weighting)
- Stratification par sexe/âge lors de l'entraînement
- Reporting de fairness (parity ratio, equal opportunity)


# 8. Export des résultats


In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder les datasets enrichis
df.to_csv('../data/processed/02_atmp_avec_biais.csv', index=False)
matrice_ratios.to_csv('../data/processed/02_matrice_biais_age.csv')
df_chi2.to_csv('../data/processed/02_tests_chi2_genre.csv', index=False)
df_biais.to_csv('../data/processed/02_biais_critiques.csv', index=False)

print("Datasets exportés dans data/processed/ :")
print("  - 02_atmp_avec_biais.csv      (dataset enrichi)")
print("  - 02_matrice_biais_age.csv    (matrice 9x4)")
print("  - 02_tests_chi2_genre.csv     (résultats Chi²)")
print("  - 02_biais_critiques.csv      (biais > 10%)")


In [ ]:
print("\n" + "=" * 60)
print("  NOTEBOOK 02 — DÉTECTION DES BIAIS TERMINÉE")
print("=" * 60)
print(f"\n  Tests statistiques effectués : 9")
print(f"  Biais significatifs (Chi²)    : {nb_biais}/9")
print(f"  Biais critiques détectés      : {len(df_biais)}")
print(f"  Figures générées              : 3")
print(f"\n  Prochaine étape : 03_correlations.ipynb")
print(f"  → Corrélations AT/MP × expositions DARES")
